<a href="https://colab.research.google.com/github/xyz111131/AI-Tools-for-Statistical-Research/blob/main/FashionMNIST_CNN_Lightning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### PyTorch Lightning Implementation
Let's install the necessary packages and rewrite the standard PyTorch code into a PyTorch Lightning module for better readability and reduced boilerplate.

In [ ]:
!pip install pytorch-lightning torchmetrics -qU

In [ ]:
import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks.early_stopping import EarlyStopping
import torchmetrics
import wandb
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

In [ ]:
class LitFashionCNN(pl.LightningModule):
    def __init__(self, learning_rate=0.0001):
        super().__init__()
        self.save_hyperparameters()
        self.learning_rate = learning_rate

        # Model Architecture (same as before)
        self.layer1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.layer2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)

        # Loss & Metrics
        self.criterion = nn.CrossEntropyLoss()
        self.train_acc = torchmetrics.Accuracy(task="multiclass", num_classes=10)
        self.val_acc = torchmetrics.Accuracy(task="multiclass", num_classes=10)
        self.test_acc = torchmetrics.Accuracy(task="multiclass", num_classes=10)

        self.test_preds = []
        self.test_targets = []

    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = out.view(out.size(0), -1)
        out = self.fc1(out)
        out = self.relu(out)
        out = self.fc2(out)
        return out

    def training_step(self, batch, batch_idx):
        images, labels = batch
        outputs = self(images)
        loss = self.criterion(outputs, labels)
        preds = torch.argmax(outputs, dim=1)

        self.train_acc(preds, labels)
        self.log("train/loss", loss, on_step=False, on_epoch=True)
        self.log("train/accuracy", self.train_acc, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        images, labels = batch
        outputs = self(images)
        loss = self.criterion(outputs, labels)
        preds = torch.argmax(outputs, dim=1)

        self.val_acc(preds, labels)
        self.log("val/loss", loss, prog_bar=True)
        self.log("val/accuracy", self.val_acc, prog_bar=True)

    def test_step(self, batch, batch_idx):
        images, labels = batch
        outputs = self(images)
        loss = self.criterion(outputs, labels)
        preds = torch.argmax(outputs, dim=1)

        self.test_acc(preds, labels)
        self.log("test/loss", loss)
        self.log("test/accuracy", self.test_acc)

        self.test_preds.append(preds)
        self.test_targets.append(labels)

    def on_test_epoch_end(self):
        # Calculate confusion matrix at the end of testing
        preds = torch.cat(self.test_preds).cpu().numpy()
        targets = torch.cat(self.test_targets).cpu().numpy()

        cm = confusion_matrix(targets, preds)
        plt.figure(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
        plt.xlabel('Predicted')
        plt.ylabel('True')
        plt.title('Confusion Matrix')

        # Log to WandB
        if isinstance(self.logger, WandbLogger):
            wandb.log({"confusion_matrix": wandb.Image(plt)})
        plt.show()

        self.test_preds.clear()
        self.test_targets.clear()

    def configure_optimizers(self):
        optimizer = optim.Adam(self.parameters(), lr=self.learning_rate)
        return optimizer

In [ ]:
class FashionMNISTDataModule(pl.LightningDataModule):
    def __init__(self, data_dir: str = "./data", batch_size: int = 64):
        super().__init__()
        self.data_dir = data_dir
        self.batch_size = batch_size
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,))
        ])

    def prepare_data(self):
        # Download data (this runs on only one GPU if distributed)
        torchvision.datasets.FashionMNIST(self.data_dir, train=True, download=True)
        torchvision.datasets.FashionMNIST(self.data_dir, train=False, download=True)

    def setup(self, stage=None):
        # Assign train/val datasets for use in dataloaders
        if stage == "fit" or stage is None:
            full_train_set = torchvision.datasets.FashionMNIST(self.data_dir, train=True, transform=self.transform)
            self.train_subset, self.val_subset = random_split(full_train_set, [50000, 10000])

        # Assign test dataset for use in dataloader(s)
        if stage == "test" or stage is None:
            self.test_set = torchvision.datasets.FashionMNIST(self.data_dir, train=False, transform=self.transform)

    def train_dataloader(self):
        return DataLoader(self.train_subset, batch_size=self.batch_size, shuffle=True, num_workers=2)

    def val_dataloader(self):
        return DataLoader(self.val_subset, batch_size=self.batch_size, shuffle=False, num_workers=2)

    def test_dataloader(self):
        return DataLoader(self.test_set, batch_size=self.batch_size, shuffle=False, num_workers=2)

In [ ]:
# Initialize Model and Callbacks
model = LitFashionCNN(learning_rate=0.0001)
datamodule = FashionMNISTDataModule(batch_size=64)

# Early stopping
early_stop_callback = EarlyStopping(monitor="val/loss", min_delta=0.00, patience=5, verbose=True, mode="min")

# WandB Logger
wandb_logger = WandbLogger(project="fashion_mnist_experiment", name="Lightning_CNN_DataModule")

# Trainer
trainer = pl.Trainer(
    max_epochs=50,
    callbacks=[early_stop_callback],
    logger=wandb_logger,
    accelerator="auto",
    devices="auto"
)

# Train and Test
trainer.fit(model, datamodule=datamodule)
trainer.test(model, datamodule=datamodule)

wandb.finish()

### Debugging and Sanity Checks
PyTorch Lightning provides built-in flags to easily debug your code and model capacity.

In [ ]:
# 1. fast_dev_run: Runs 1 batch of train, val, and test to catch bugs quickly
print("--- Running fast_dev_run ---")
debug_trainer = pl.Trainer(fast_dev_run=True)
# Uncomment to test:
debug_trainer.fit(model, datamodule=datamodule)

# 2. overfit_batches: Trains on a single batch to ensure the model can memorize it
print("\n--- Running overfit_batches ---")
# Use a small number of epochs to see the loss drop quickly on that single batch
overfit_trainer = pl.Trainer(overfit_batches=1, max_epochs=5) #use the same single batch for training and validation.
# Uncomment to test:
overfit_trainer.fit(model, datamodule=datamodule)

### Profiling
Find bottlenecks in your data loading or training steps using Lightning's built-in profilers.

In [ ]:
# Pass profiler="simple", "advanced", or "pytorch" to the Trainer
print("--- Running Profiler ---")

# We'll use fast_dev_run=5 to quickly simulate 5 batches and generate a profile report
profile_trainer = pl.Trainer(
    fast_dev_run=5,
    profiler="simple"  # Try changing this to "advanced" or "pytorch"
)

# Uncomment to run the profiler and print the report:
profile_trainer.fit(model, datamodule=datamodule)